In [1]:
import numpy as np
import cv2
from collections import defaultdict
from ultralytics import YOLO

# ── VISEM 촬영 조건 기반 변환 상수 ─────────────────────────
# Olympus CX31, 400x 배율, 640x480 해상도
# 400x 위상차 현미경의 표준 FOV ≈ 450µm × 338µm
# → 1 pixel = 450/640 ≈ 0.703 µm/px

UM_PER_PX = 450.0 / 640.0   # µm/pixel
FPS       = 50.0             # frames/second

print(f"변환 상수:")
print(f"  1 pixel = {UM_PER_PX:.4f} µm")
print(f"  FPS     = {FPS} fps")
print(f"  → 1 px/frame = {UM_PER_PX * FPS:.2f} µm/s")

# ── CASA 키네마틱 파라미터 계산 함수 ──────────────────────
def compute_casa_kinematics(coords: np.ndarray,
                             fps: float = FPS,
                             um_per_px: float = UM_PER_PX) -> dict:
    """
    정자 이동 좌표 → CASA 표준 키네마틱 파라미터 계산
    
    Args:
        coords: (N, 2) 배열 [(x1,y1), (x2,y2), ...]  단위: pixel
        fps:    프레임레이트
        um_per_px: 픽셀당 마이크로미터
    
    Returns:
        VCL, VSL, VAP, LIN, STR, WOB (µm/s 또는 무차원)
    """
    if len(coords) < 3:
        return None

    n = len(coords)
    dt = 1.0 / fps  # 프레임 간격 (초)

    # ── VCL: Curvilinear Velocity (실제 경로 속도) ──────────
    # 연속 프레임 간 거리의 합 / 총 시간
    dists = np.sqrt(np.sum(np.diff(coords, axis=0)**2, axis=1))
    total_path_px = float(np.sum(dists))
    total_time    = (n - 1) * dt
    VCL = (total_path_px * um_per_px) / total_time  # µm/s

    # ── VSL: Straight-Line Velocity (직선 속도) ─────────────
    # 시작점~끝점 직선 거리 / 총 시간
    straight_dist_px = float(np.sqrt(
        (coords[-1][0] - coords[0][0])**2 +
        (coords[-1][1] - coords[0][1])**2))
    VSL = (straight_dist_px * um_per_px) / total_time  # µm/s

    # ── VAP: Average Path Velocity (평균 경로 속도) ──────────
    # 이동 경로를 5점 이동 평균으로 스무딩 후 속도 계산
    window = min(5, n)
    smoothed = np.array([
        coords[max(0, i-window//2):i+window//2+1].mean(axis=0)
        for i in range(n)
    ])
    smooth_dists  = np.sqrt(np.sum(np.diff(smoothed, axis=0)**2, axis=1))
    avg_path_px   = float(np.sum(smooth_dists))
    VAP = (avg_path_px * um_per_px) / total_time  # µm/s

    # ── 비율 파라미터 (무차원) ───────────────────────────────
    LIN = VSL / (VCL + 1e-6)          # 직선성 = VSL/VCL
    STR = VSL / (VAP + 1e-6)          # 직진성 = VSL/VAP
    WOB = VAP / (VCL + 1e-6)          # 진동도 = VAP/VCL

    # ── ALH: Amplitude of Lateral Head Displacement ──────────
    # 실제 경로와 평균 경로 간의 횡방향 편차 평균
    lateral_devs = []
    for i in range(n):
        dev = np.sqrt(np.sum((coords[i] - smoothed[i])**2))
        lateral_devs.append(dev)
    ALH = float(np.mean(lateral_devs)) * um_per_px  # µm

    # ── BCF: Beat Cross Frequency ──────────────────────────
    # 실제 경로가 평균 경로를 교차하는 빈도 (Hz)
    # 평균 경로 기준 좌우 부호 변화 횟수 계산
    cross_count = 0
    for i in range(1, n):
        v1 = coords[i-1] - smoothed[i-1]
        v2 = coords[i]   - smoothed[i]
        cross_prod = v1[0]*v2[1] - v1[1]*v2[0]
        if i > 1:
            v0 = coords[i-2] - smoothed[i-2]
            prev = v0[0]*v1[1] - v0[1]*v1[0]
            if cross_prod * prev < 0:
                cross_count += 1
    BCF = cross_count / total_time  # Hz

    return {
        'VCL': round(VCL, 2),   # µm/s  (정상 > 25 µm/s)
        'VSL': round(VSL, 2),   # µm/s  (정상 > 15 µm/s)
        'VAP': round(VAP, 2),   # µm/s  (정상 > 20 µm/s)
        'LIN': round(LIN, 3),   # 0~1   (정상 > 0.5)
        'STR': round(STR, 3),   # 0~1   (정상 > 0.8)
        'WOB': round(WOB, 3),   # 0~1
        'ALH': round(ALH, 2),   # µm
        'BCF': round(BCF, 2),   # Hz
    }

print("\n✅ CASA 키네마틱 계산 함수 준비 완료")

# ── 테스트: 가상의 정자 경로로 검증 ───────────────────────
print("\n=== 운동 유형별 테스트 ===")

# 1. 전진 운동성 (직선 이동)
t = np.linspace(0, 3, 150)
prog_coords = np.column_stack([t * 20, t * 5 + np.sin(t*10)*2])
r = compute_casa_kinematics(prog_coords)
print(f"\n전진 운동성 (직선):")
print(f"  VCL={r['VCL']} VSL={r['VSL']} VAP={r['VAP']} µm/s")
print(f"  LIN={r['LIN']} STR={r['STR']} WOB={r['WOB']}")

# 2. 비전진 운동성 (원운동)
angle = np.linspace(0, 4*np.pi, 150)
nonprog_coords = np.column_stack([
    np.cos(angle)*15 + 320,
    np.sin(angle)*15 + 240
])
r = compute_casa_kinematics(nonprog_coords)
print(f"\n비전진 운동성 (원운동):")
print(f"  VCL={r['VCL']} VSL={r['VSL']} VAP={r['VAP']} µm/s")
print(f"  LIN={r['LIN']} STR={r['STR']} WOB={r['WOB']}")

# 3. 비운동성 (제자리)
np.random.seed(42)
still_coords = np.random.randn(150, 2) * 0.5 + np.array([320, 240])
r = compute_casa_kinematics(still_coords)
print(f"\n비운동성 (정지):")
print(f"  VCL={r['VCL']} VSL={r['VSL']} VAP={r['VAP']} µm/s")
print(f"  LIN={r['LIN']} STR={r['STR']} WOB={r['WOB']}")

변환 상수:
  1 pixel = 0.7031 µm
  FPS     = 50.0 fps
  → 1 px/frame = 35.16 µm/s

✅ CASA 키네마틱 계산 함수 준비 완료

=== 운동 유형별 테스트 ===

전진 운동성 (직선):
  VCL=17.37 VSL=14.49 VAP=16.92 µm/s
  LIN=0.834 STR=0.856 WOB=0.974

비전진 운동성 (원운동):
  VCL=44.46 VSL=0.0 VAP=43.56 µm/s
  LIN=0.0 STR=0.0 WOB=0.98

비운동성 (정지):
  VCL=31.7 VSL=0.1 VAP=6.11 µm/s
  LIN=0.003 STR=0.016 WOB=0.193


In [2]:
import sys
sys.path.append(r'C:\Users\neo62\sperm-ai')
from src.pipeline import SpermAnalysisPipeline
from collections import defaultdict
import numpy as np
import cv2

# 모델 로드
pipeline = SpermAnalysisPipeline()
config_path = r'C:\Users\neo62\sperm-ai\bytetrack_custom.yaml'

# ── 참가자 11번으로 실제 키네마틱 계산 ────────────────────
video_path = r'C:\Users\neo62\sperm-ai\data\raw\VISEM-Tracking\VISEM_Tracking_Train_v4\Train\11\11.mp4'

cap = cv2.VideoCapture(video_path)
track_history = defaultdict(list)
fps_video = cap.get(cv2.CAP_PROP_FPS)
max_frames = min(int(fps_video * 5), 250)

for fidx in range(max_frames):
    ret, frame = cap.read()
    if not ret: break
    res = pipeline.detector.model.track(
        frame, persist=True,
        tracker=config_path,
        verbose=False, conf=0.3)
    if res[0].boxes.id is not None:
        for box, tid, cls in zip(
            res[0].boxes.xywh.cpu().numpy(),
            res[0].boxes.id.cpu().numpy().astype(int),
            res[0].boxes.cls.cpu().numpy().astype(int)
        ):
            if cls == 0:
                track_history[tid].append(
                    (fidx, float(box[0]), float(box[1])))
cap.release()

# ── 각 정자에 CASA 키네마틱 계산 ──────────────────────────
casa_results = []
for tid, pts in track_history.items():
    if len(pts) < 10:
        continue
    coords = np.array([(cx, cy) for _, cx, cy in pts])
    k = compute_casa_kinematics(coords, fps=fps_video)
    if k:
        k['tid'] = tid
        k['n_frames'] = len(pts)
        casa_results.append(k)

# ── WHO 기준으로 분류 ───────────────────────────────────
# 전진: VSL > 15 µm/s AND STR > 0.8
# 비전진: VCL > 5 µm/s AND (VSL < 15 OR STR < 0.8)
# 비운동: VCL < 5 µm/s OR LIN < 0.1

progressive     = [r for r in casa_results
                   if r['VSL'] >= 15 and r['STR'] >= 0.8]
non_progressive = [r for r in casa_results
                   if r['VCL'] >= 5
                   and not (r['VSL'] >= 15 and r['STR'] >= 0.8)]
immotile        = [r for r in casa_results
                   if r['VCL'] < 5 or r['LIN'] < 0.1]

total = len(progressive) + len(non_progressive) + len(immotile)

print("=== 참가자 11번 CASA 키네마틱 분석 ===\n")
print(f"분석된 정자 수: {len(casa_results)}개\n")

# 전체 평균
all_vcl = np.mean([r['VCL'] for r in casa_results])
all_vsl = np.mean([r['VSL'] for r in casa_results])
all_vap = np.mean([r['VAP'] for r in casa_results])
all_lin = np.mean([r['LIN'] for r in casa_results])
all_str = np.mean([r['STR'] for r in casa_results])
all_wob = np.mean([r['WOB'] for r in casa_results])
all_alh = np.mean([r['ALH'] for r in casa_results])

print("=== 샘플 평균 키네마틱 파라미터 ===")
print(f"  VCL: {all_vcl:.1f} µm/s  (정상 기준 > 25)")
print(f"  VSL: {all_vsl:.1f} µm/s  (정상 기준 > 15)")
print(f"  VAP: {all_vap:.1f} µm/s  (정상 기준 > 20)")
print(f"  LIN: {all_lin:.3f}       (정상 기준 > 0.50)")
print(f"  STR: {all_str:.3f}       (정상 기준 > 0.80)")
print(f"  WOB: {all_wob:.3f}")
print(f"  ALH: {all_alh:.2f} µm")

print(f"\n=== WHO 기준 운동성 분류 ===")
print(f"  전진 운동성:   {len(progressive):3d}개 "
      f"({len(progressive)/len(casa_results)*100:.1f}%)")
print(f"  비전진 운동성: {len(non_progressive):3d}개 "
      f"({len(non_progressive)/len(casa_results)*100:.1f}%)")
print(f"  비운동성:      {len(immotile):3d}개 "
      f"({len(immotile)/len(casa_results)*100:.1f}%)")

print(f"\n=== 실제 검사 결과 (참가자 11) ===")
print(f"  전진 운동성:   11%")
print(f"  비전진 운동성: 17%")
print(f"  비운동성:      72%")

✅ SpermDetector 로드 완료: C:\Users\neo62\sperm-ai\models\yolo11_sperm_v2\weights\best.pt
✅ MotilityAnalyzer v1 로드 완료: C:\Users\neo62\sperm-ai\models\motility_ensemble.pkl
   보정 레이어: 없음 (v1)
=== 참가자 11번 CASA 키네마틱 분석 ===

분석된 정자 수: 78개

=== 샘플 평균 키네마틱 파라미터 ===
  VCL: 39.8 µm/s  (정상 기준 > 25)
  VSL: 21.4 µm/s  (정상 기준 > 15)
  VAP: 30.0 µm/s  (정상 기준 > 20)
  LIN: 0.331       (정상 기준 > 0.50)
  STR: 0.458       (정상 기준 > 0.80)
  WOB: 0.662
  ALH: 0.40 µm

=== WHO 기준 운동성 분류 ===
  전진 운동성:    12개 (15.4%)
  비전진 운동성:  48개 (61.5%)
  비운동성:       32개 (41.0%)

=== 실제 검사 결과 (참가자 11) ===
  전진 운동성:   11%
  비전진 운동성: 17%
  비운동성:      72%


In [3]:
import numpy as np

# ── WHO 기준 기반 상호 배타적 분류 ───────────────────────
# 우선순위: 비운동성 먼저 판별 → 전진 → 비전진

def classify_sperm_who(k: dict) -> str:
    """
    WHO 6판 기준으로 정자 운동성 분류
    상호 배타적: 하나의 정자는 하나의 카테고리만
    
    기준:
    - 비운동성: VCL < 8 µm/s (거의 안 움직임)
    - 전진:     VSL ≥ 15 µm/s AND STR ≥ 0.6
    - 비전진:   그 외 (움직이지만 전진 못함)
    """
    # 1순위: 비운동성 판별
    if k['VCL'] < 8.0:
        return 'immotile'

    # 2순위: 전진 운동성 판별
    if k['VSL'] >= 15.0 and k['STR'] >= 0.6:
        return 'progressive'

    # 3순위: 비전진 운동성
    return 'non_progressive'

# ── 분류 재실행 ──────────────────────────────────────────
prog_list = []
non_prog_list = []
immotile_list = []

for r in casa_results:
    cls = classify_sperm_who(r)
    if cls == 'progressive':
        prog_list.append(r)
    elif cls == 'non_progressive':
        non_prog_list.append(r)
    else:
        immotile_list.append(r)

total = len(casa_results)

print("=== 수정된 분류 결과 ===")
print(f"분석 정자 수: {total}개\n")
print(f"전진 운동성:   {len(prog_list):3d}개 "
      f"({len(prog_list)/total*100:.1f}%)")
print(f"비전진 운동성: {len(non_prog_list):3d}개 "
      f"({len(non_prog_list)/total*100:.1f}%)")
print(f"비운동성:      {len(immotile_list):3d}개 "
      f"({len(immotile_list)/total*100:.1f}%)")
print(f"합계:          {total}개 (100%)")

print(f"\n=== 실제 검사 결과 ===")
print(f"전진 운동성:   11%")
print(f"비전진 운동성: 17%")
print(f"비운동성:      72%")

print(f"\n=== 전진 운동성 정자 키네마틱 평균 ===")
if prog_list:
    print(f"  VCL: {np.mean([r['VCL'] for r in prog_list]):.1f} µm/s")
    print(f"  VSL: {np.mean([r['VSL'] for r in prog_list]):.1f} µm/s")
    print(f"  STR: {np.mean([r['STR'] for r in prog_list]):.3f}")

print(f"\n=== 비운동성 정자 키네마틱 평균 ===")
if immotile_list:
    print(f"  VCL: {np.mean([r['VCL'] for r in immotile_list]):.1f} µm/s")
    print(f"  LIN: {np.mean([r['LIN'] for r in immotile_list]):.3f}")

# ── 임계값 분석 ─────────────────────────────────────────
print(f"\n=== VCL 분포 분석 ===")
vcl_vals = sorted([r['VCL'] for r in casa_results])
print(f"  최솟값: {vcl_vals[0]:.1f}")
print(f"  25%:    {np.percentile(vcl_vals, 25):.1f}")
print(f"  중앙값: {np.median(vcl_vals):.1f}")
print(f"  75%:    {np.percentile(vcl_vals, 75):.1f}")
print(f"  최댓값: {vcl_vals[-1]:.1f}")

print(f"\n  VCL < 8  µm/s (비운동성): "
      f"{sum(1 for v in vcl_vals if v < 8)}개 "
      f"({sum(1 for v in vcl_vals if v < 8)/len(vcl_vals)*100:.1f}%)")
print(f"  VCL < 15 µm/s:             "
      f"{sum(1 for v in vcl_vals if v < 15)}개 "
      f"({sum(1 for v in vcl_vals if v < 15)/len(vcl_vals)*100:.1f}%)")
print(f"  VCL < 25 µm/s:             "
      f"{sum(1 for v in vcl_vals if v < 25)}개 "
      f"({sum(1 for v in vcl_vals if v < 25)/len(vcl_vals)*100:.1f}%)")

=== 수정된 분류 결과 ===
분석 정자 수: 78개

전진 운동성:    17개 (21.8%)
비전진 운동성:  27개 (34.6%)
비운동성:       34개 (43.6%)
합계:          78개 (100%)

=== 실제 검사 결과 ===
전진 운동성:   11%
비전진 운동성: 17%
비운동성:      72%

=== 전진 운동성 정자 키네마틱 평균 ===
  VCL: 97.1 µm/s
  VSL: 66.9 µm/s
  STR: 0.939

=== 비운동성 정자 키네마틱 평균 ===
  VCL: 4.8 µm/s
  LIN: 0.161

=== VCL 분포 분석 ===
  최솟값: 1.9
  25%:    5.3
  중앙값: 9.7
  75%:    65.0
  최댓값: 163.3

  VCL < 8  µm/s (비운동성): 34개 (43.6%)
  VCL < 15 µm/s:             43개 (55.1%)
  VCL < 25 µm/s:             44개 (56.4%)


In [1]:
import sys
sys.path.append(r'C:\Users\neo62\sperm-ai')
from src.pipeline import SpermAnalysisPipeline

pipeline = SpermAnalysisPipeline()

result = pipeline.analyze(
    r'C:\Users\neo62\sperm-ai\data\raw\VISEM-Tracking\VISEM_Tracking_Train_v4\Train\11\11.mp4'
)

pipeline.print_report(result, participant_id='11')

✅ SpermDetector 로드 완료: C:\Users\neo62\sperm-ai\models\yolo11_sperm_v2\weights\best.pt
✅ MotilityAnalyzer v1 로드 완료: C:\Users\neo62\sperm-ai\models\motility_ensemble.pkl
   보정 레이어: 없음 (v1)
분석 시작: 11.mp4
──────────────────────────────────────────────────
[1/4] 정자 수 기준값: 50개
[2/4] 추적 완료: 108개 ID
[3/4] 특징 추출 완료
[4/4] 분석 완료

  정자 운동성 분석 보고서 — 참가자 11

📊 분석 신뢰도: ✅ 높음 (100점)

📈 운동성 분석 결과 (탐지 정자 수: 50개)
   전진 운동성:    21.4%
   비전진 운동성:  26.2%
   비운동성:       52.4%
   총 운동성:      47.6%

🔍 WHO 기준 해석
   🔴 전진 운동성 21.4% — WHO 정상 기준보다 현저히 낮음
   ✅ 총 운동성 47.6% — WHO 정상 기준(40%) 충족
   ⚠️  비운동성 52.4% — 주의 필요 (기준 50% 초과)

🔴 종합 판정: [주의 필요]

💬 권고사항:
   AI 분석 결과 일부 지표가 WHO 기준 이하입니다. 정밀 검사를 위해 병원 방문을 권고합니다. 이 결과는 보조 분석이며 의학적 진단을 대체하지 않습니다.

📐 CASA 키네마틱 파라미터
   VCL:   40.6 µm/s  VSL:   21.9 µm/s  VAP:   30.6 µm/s
   LIN:  0.331       STR:  0.458       WOB:  0.662
   ALH:   0.40 µm  (분석 정자: 78개)

───────────────────────────────────────────────────────
   ※ 이 결과는 AI 보조 분석이며 의학적 진단이 아닙니다.



In [1]:
import sys
sys.path.append(r'C:\Users\neo62\sperm-ai')
from src.pipeline import SpermAnalysisPipeline

pipeline = SpermAnalysisPipeline()

result = pipeline.analyze(
    r'C:\Users\neo62\sperm-ai\data\raw\VISEM-Tracking\VISEM_Tracking_Train_v4\Train\11\11.mp4'
)

pipeline.print_report(result, participant_id='11')

✅ SpermDetector 로드 완료: C:\Users\neo62\sperm-ai\models\yolo11_sperm_v2\weights\best.pt
✅ MotilityAnalyzer v1 로드 완료: C:\Users\neo62\sperm-ai\models\motility_ensemble.pkl
   보정 레이어: 없음 (v1)
분석 시작: 11.mp4
──────────────────────────────────────────────────
[1/4] 정자 수 기준값: 50개
[2/4] 추적 완료: 108개 ID
[3/4] 특징 추출 완료
[4/4] 분석 완료

  정자 운동성 분석 보고서 — 참가자 11

📊 분석 신뢰도: ✅ 높음 (100점)

📈 운동성 분석 결과 (탐지 정자 수: 50개)
   전진 운동성:    21.4%
   비전진 운동성:  26.2%
   비운동성:       52.4%
   총 운동성:      47.6%

🔍 WHO 기준 해석
   🔴 전진 운동성 21.4% — WHO 정상 기준보다 현저히 낮음
   ✅ 총 운동성 47.6% — WHO 정상 기준(40%) 충족
   ⚠️  비운동성 52.4% — 주의 필요 (기준 50% 초과)

🔴 종합 판정: [주의 필요]

💬 권고사항:
   AI 분석 결과 일부 지표가 WHO 기준 이하입니다. 정밀 검사를 위해 병원 방문을 권고합니다. 이 결과는 보조 분석이며 의학적 진단을 대체하지 않습니다.

📐 CASA 키네마틱 파라미터
   파라미터            측정값  참고 기준
   ----------------------------------------
   VCL         40.6 µm/s  ≥ 25 µm/s
   VSL         21.9 µm/s  ≥ 15 µm/s
   VAP         30.6 µm/s  ≥ 20 µm/s
   LIN        0.331       ≥ 0.50
   STR        0.458       ≥ 0.80
   WOB      